# Clima NASA POWER mensual (2002–2025)

Esta notebook reutiliza los JSON diarios de NASA POWER y genera un dataset climático mensual por municipio. Se incluye 2002 para permitir ventanas agrícolas que comienzan antes de 2003. Los archivos permiten reanudar una interrupción.

Cada fila representa un municipio, año y mes. La temperatura se promedia y la precipitación se acumula utilizando únicamente meses con todos sus días disponibles.

In [1]:
import logging

import pandas as pd

from proyecto_agronomia.config import (
    NASA_POWER_ANIO_FINAL,
    NASA_POWER_ANIO_INICIAL,
    NASA_POWER_PROCESSED_DIR,
)
from proyecto_agronomia.datasets.clima_nasa import (
    MESES,
    descargar_clima_nasa_mensual_por_celdas,
)
from proyecto_agronomia.datasets.inegi import obtener_municipios_inegi
from proyecto_agronomia.processing.integracion_climatica import (
    puntos_representativos_municipales,
)

ANIO_INICIAL = NASA_POWER_ANIO_INICIAL
ANIO_FINAL = NASA_POWER_ANIO_FINAL
RUTA_SALIDA = (
    NASA_POWER_PROCESSED_DIR
    / f"clima_municipal_mensual_{ANIO_INICIAL - 1}_{ANIO_FINAL}.csv"
)
logging.basicConfig(level=logging.INFO)
pd.set_option("display.max_columns", None)

2026-10-29 18:12:20.313 | INFO     | proyecto_agronomia.config:<module>:14 - PROJ_ROOT path is: B:\MCD2\JW\agro\proyecto_agronomia


## 1. Preparar puntos municipales de INEGI

In [2]:
gdf_municipios_inegi = obtener_municipios_inegi()
df_puntos_nasa = puntos_representativos_municipales(gdf_municipios_inegi)

df_municipios_nasa = (
    df_puntos_nasa[["cvegeo", "nomgeo", "latitud", "longitud"]]
    .rename(columns={
        "nomgeo": "Nommunicipio",
        "latitud": "Latitud",
        "longitud": "Longitud",
    })
    .sort_values("cvegeo")
    .reset_index(drop=True)
)
df_municipios_nasa["cvegeo"] = df_municipios_nasa["cvegeo"].astype("string").str.zfill(5)

assert df_municipios_nasa["cvegeo"].is_unique
assert df_municipios_nasa[["Latitud", "Longitud"]].notna().all().all()
print(f"Municipios preparados: {len(df_municipios_nasa):,}")
display(df_municipios_nasa.head())

2026-10-29 18:12:24.771 | INFO     | proyecto_agronomia.datasets.inegi:obtener_municipios_inegi:29 - Cargando municipios INEGI desde archivo local.
Municipios preparados: 2,478


,cvegeo,Nommunicipio,Latitud,Longitud
0,01001,Aguascalientes,21.848487,-102.305449
1,01002,Asientos,22.132227,-102.095219
2,01003,Calvillo,21.911174,-102.713168
3,01004,Cosío,22.371703,-102.311445
4,01005,Jesús María,21.928961,-102.461647


## 2. Generar el clima mensual

Se reutilizan 2003–2022 por celda y 2023–2025 por municipio. Los JSON de 2002 descargados en la ejecución anterior también se reutilizan, por lo que esta etapa consiste principalmente en reagregar los datos diarios.

In [3]:
EJECUTAR_AGREGACION_MENSUAL = True

if EJECUTAR_AGREGACION_MENSUAL:
    df_clima_mensual = descargar_clima_nasa_mensual_por_celdas(
        municipios=df_municipios_nasa,
        inicio=ANIO_INICIAL,
        fin=ANIO_FINAL,
    )
    print(f"Registros generados: {len(df_clima_mensual):,}")
else:
    print("Descarga omitida; se utilizará el CSV existente.")

INFO:proyecto_agronomia.datasets.clima_nasa:NASA POWER por periodos 2003-2025: 2478 municipios en 476 celdas
INFO:proyecto_agronomia.datasets.clima_nasa:Procesadas 25/476 celdas
INFO:proyecto_agronomia.datasets.clima_nasa:Procesadas 50/476 celdas
INFO:proyecto_agronomia.datasets.clima_nasa:Procesadas 75/476 celdas
INFO:proyecto_agronomia.datasets.clima_nasa:Procesadas 100/476 celdas
INFO:proyecto_agronomia.datasets.clima_nasa:Procesadas 125/476 celdas
INFO:proyecto_agronomia.datasets.clima_nasa:Procesadas 150/476 celdas
INFO:proyecto_agronomia.datasets.clima_nasa:Procesadas 175/476 celdas
INFO:proyecto_agronomia.datasets.clima_nasa:Procesadas 200/476 celdas
INFO:proyecto_agronomia.datasets.clima_nasa:Procesadas 225/476 celdas
INFO:proyecto_agronomia.datasets.clima_nasa:Procesadas 250/476 celdas
INFO:proyecto_agronomia.datasets.clima_nasa:Procesadas 275/476 celdas
INFO:proyecto_agronomia.datasets.clima_nasa:Procesadas 300/476 celdas
INFO:proyecto_agronomia.datasets.clima_nasa:Procesadas

Registros generados: 713,664


## 3. Cargar y validar el archivo mensual

In [4]:
if not RUTA_SALIDA.exists():
    raise FileNotFoundError(
        f"No existe {RUTA_SALIDA}. Activa EJECUTAR_AGREGACION_MENSUAL."
    )

df_clima_mensual = pd.read_csv(RUTA_SALIDA, dtype={"cvegeo": "string"})
df_clima_mensual["cvegeo"] = df_clima_mensual["cvegeo"].str.zfill(5)

claves_esperadas = set(df_municipios_nasa["cvegeo"])
anios_esperados = set(range(ANIO_INICIAL - 1, ANIO_FINAL + 1))
meses_esperados = set(MESES)
filas_esperadas = (
    len(claves_esperadas) * len(anios_esperados) * len(meses_esperados)
)
clave = ["cvegeo", "anio", "mes"]

assert not df_clima_mensual.duplicated(clave).any()
assert set(df_clima_mensual["cvegeo"]) == claves_esperadas
assert set(df_clima_mensual["anio"]) == anios_esperados
assert set(df_clima_mensual["mes"]) == meses_esperados
assert len(df_clima_mensual) == filas_esperadas
assert df_clima_mensual[[
    "temperatura_media_mes_c",
    "precipitacion_mes_mm",
]].notna().all().all()
assert (
    df_clima_mensual["dias_temperatura"]
    == df_clima_mensual["dias_esperados"]
).all()
assert (
    df_clima_mensual["dias_precipitacion"]
    == df_clima_mensual["dias_esperados"]
).all()

resumen = df_clima_mensual.groupby(["anio", "mes"]).agg(
    municipios=("cvegeo", "nunique"),
    temperatura_media_c=("temperatura_media_mes_c", "mean"),
    precipitacion_media_mm=("precipitacion_mes_mm", "mean"),
)
display(resumen)
display(df_clima_mensual.head())
print(f"Archivo validado: {RUTA_SALIDA}")
print(f"Filas: {len(df_clima_mensual):,}/{filas_esperadas:,}")

municipios  temperatura_media_c  precipitacion_media_mm
anio mes                                                         
2002 1          2478            15.858325               13.350375
     2          2478            16.524126               26.698971
     3          2478            19.751880                8.022623
     4          2478            22.584898               10.670024
     5          2478            23.565304               36.407135
...              ...                  ...                     ...
2025 8          2478            22.004512              158.556324
     9          2478            21.552596              197.195153
     10         2478            20.128518              159.819726
     11         2478            18.514867               34.656416
     12         2478            17.913896               44.528834

[288 rows x 3 columns]

,cvegeo,anio,mes,fecha_inicio,fecha_fin,temperatura_media_mes_c,precipitacion_mes_mm,dias_esperados,dias_temperatura,dias_precipitacion,fuente,celda_lat,celda_lon
0,01001,2002,1,2002-01-01,2002-01-31,12.089355,17.34,31,31,31,NASA POWER,22.0,-102.5
1,01001,2002,2,2002-02-01,2002-02-28,13.223929,29.07,28,28,28,NASA POWER,22.0,-102.5
2,01001,2002,3,2002-03-01,2002-03-31,16.687097,0.04,31,31,31,NASA POWER,22.0,-102.5
3,01001,2002,4,2002-04-01,2002-04-30,20.270000,2.60,30,30,30,NASA POWER,22.0,-102.5
4,01001,2002,5,2002-05-01,2002-05-31,22.468387,24.57,31,31,31,NASA POWER,22.0,-102.5


Archivo validado: B:\MCD2\JW\agro\proyecto_agronomia\data\processed\clima_nasa\clima_municipal_mensual_2002_2025.csv
Filas: 713,664/713,664
